soundfonts:

https://sites.google.com/site/soundfonts4u/

In [1]:
from utils.pixelated import pixelate
pixelate()

In [2]:
import threading
from queue import PriorityQueue
import time

class StoppableScheduler:
    def __init__(self):
        self.queue = PriorityQueue()
        self.thread = None
        self.running = False
        self.condition = threading.Condition()

    def enter(self, time, priority, action, argument=(), kwargs={}):
        self.queue.put((time, priority, action, argument, kwargs))

    def run(self):
        self.thread = threading.Thread(target=self.threaded_job)
        self.running = True
        self.thread.start()

    def threaded_job(self):
        start_time = time.monotonic()
        while self.running and not self.queue.empty():
            cur_time = time.monotonic()
            next_time, _, action, argument, kwargs = self.queue.get()
            delay_duration = (start_time + next_time) - cur_time
            if delay_duration > 0:
                with self.condition:
                    self.condition.wait(delay_duration)
            if self.running:
                action(*argument, **kwargs)

    def stop(self):
        if self.thread:
            self.running = False
            with self.condition:
                self.condition.notify()
            self.thread.join()
            self.queue = PriorityQueue()

In [3]:
import time
import sched
import threading
from collections import defaultdict

from utils.imutil import imshow
from utils.list_files import list_files

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Button, HBox
import rtmidi
import pretty_midi
from IPython.display import display
from IPython.display import clear_output

midiout = rtmidi.MidiOut()
midiout.open_virtual_port("Music Transformer")


In [4]:
diatonic = [1,0,1,0,1,1,0,1,0,1,0,1]
keys = []
for i in range(12):
    keys.append(np.roll(diatonic, i))
keys = np.asarray(keys)

def get_key(notes):
    pitches = [e.pitch for e in notes]
    counts, _ = np.histogram(np.asarray(pitches) % 12, bins=12, range=(0,12))
    non_diatonic = ((1-keys) * counts).sum(1)
    return {
        'key': non_diatonic.argmin(),
        'accidental_count': non_diatonic.min()
    }

key_names = """\
C major, A minor
C♯/D♭ major, B♭ minor
D major, B minor
E♭ major, C minor
E major, C♯ minor
F major, D minor
G♭/F♯ major, E♭/D♯ minor
G major, E minor
A♭ major, F minor
A major, F♯ minor
B♭ major, G minor
B/C♭ major, G♯ minor""".splitlines()

def get_intervals(chord):
    return np.diff(sorted(chord))

def notes_to_events(notes):
    events = []
    for e in notes:
        events.append((e.start, e.pitch, e.velocity))
        events.append((e.end, e.pitch, 0))
    events.sort()
    return events

def get_interval_count(events):
    counts = defaultdict(float)
    on_notes = set()
    previous_t = 0
    polyphony = []
    for t,p,v in events:
        duration = t - previous_t
        intervals = get_intervals(on_notes)
        polyphony.append(len(on_notes))
        for e in intervals:
            counts[e % 12] += duration
        if v > 0:
            on_notes.add(p)
        elif p in on_notes:
            on_notes.remove(p)
        previous_t = t
    return {
        'interval_counts': counts,
        'max_polyphony': np.max(polyphony),
        'median_polyphony': np.median(polyphony)
    }

interval_quality = {
    0: None, # u/o
    1: 'minor', # m2
    2: 'major', # M2
    3: 'minor', # m3
    4: 'major', # M3
    5: None, # P4
    6: None, # TT
    7: None, # P5
    8: 'minor', # m6
    9: 'major', # M6
    10: 'minor', # m7
    11: 'major' # M7
}

def guess_mode(notes):
    events = notes_to_events(notes)
    interval_counts = get_interval_count(events)
    quality_count = defaultdict(float)
    for interval, count in interval_counts['interval_counts'].items():
        quality_count[interval_quality[interval]] += count
    if None in quality_count:
        del quality_count[None]
    if len(quality_count) == 0:
        return None
    mode = max(quality_count, key=quality_count.get)
    del interval_counts['interval_counts']
    interval_counts['mode'] = mode
    return interval_counts

def guess_metadata(notes):
    key_metadata = get_key(notes)
    key = key_metadata['key']

    key_name = key_names[key]
    mode = guess_mode(notes)

    # i don't think this really works
    # if mode['mode'] == 'minor':
    #     key_name = key_name.split(', ')[1]
    # elif mode['mode'] == 'major':
    #     key_name = key_name.split(', ')[0]

    velocities = np.asarray([e.velocity for e in notes])
    duration = max([e.end for e in notes])
    notes_per_second = len(notes) / duration

    return {
        'key': key,
        'note_count': len(notes),
        'mode': mode['mode'],
        'max_polyphony': mode['max_polyphony'],
        'median_polyphony': mode['median_polyphony'],
        'accidental_count': key_metadata['accidental_count'],
        'key_name': key_name,
        'duration': duration,
        'velocity_median': np.median(velocities),
        'velocity_std': np.std(velocities),
        'notes_per_second': notes_per_second
    }

def print_metadata(metadata):
    print(f"key index: {metadata['key']}")
    print(f"key name: {metadata['key_name']}")
    # print(f"mode: {metadata['mode']}")
    print(f"max_polyphony: {metadata['max_polyphony']}")
    print(f"median_polyphony: {metadata['median_polyphony']}")
    print(f"accidental count: {metadata['accidental_count']}")
    print(f"duration: {metadata['duration']:.2f}s")
    print(f"median velocity: {metadata['velocity_median']:.0f}")
    print(f"std velocity: {metadata['velocity_std']:.2f}")
    print(f"notes_per_second: {metadata['notes_per_second']:.2f}")

def get_piano_roll(pm, roll_width=1024):
    duration = pm.get_end_time()
    roll = np.zeros((128, roll_width))
    for instrument in pm.instruments:
        for note in sorted(instrument.notes, key=lambda n: n.start):
            x_start = int(roll_width * note.start / duration)
            x_end = int(roll_width * note.end / duration)
            x_end = max(x_start + 1, x_end)
            roll[note.pitch, x_start:x_end] = np.maximum(roll[note.pitch, x_start:x_end],
                                                        np.linspace(note.velocity, 0, x_end - x_start,
                                                        endpoint=False))
    return roll

pr_width = 2048
def show_pr(pm):
    notes = pm.instruments[0].notes
    metadata = guess_metadata(notes)
    print_metadata(metadata)
    mask = np.tile(keys[metadata['key']], 11)[:128]
    pr = get_piano_roll(pm, pr_width)
    pr /= pr.max()
    diatonic_only = pr * mask.reshape(-1,1)
    pr = np.dstack((pr, diatonic_only, pr))
    imshow(np.flipud(pr) * 255, zoom=3)
    return pr

In [5]:
def trim_pretty_midi(pm):
    start_time = min(
        [note.start for instrument in pm.instruments for note in instrument.notes])
    for i,instrument in enumerate(pm.instruments):
        for n,note in enumerate(instrument.notes):
            pm.instruments[i].notes[n].start -= start_time
            pm.instruments[i].notes[n].end -= start_time

class ThreadedPlayer:
    def __init__(self):
        self.thread = None
        self.pm = None
        self.events = []
        self.schedule = StoppableScheduler()

    def play(self, fn):
        self.stop()
        self.pm = pretty_midi.PrettyMIDI(fn)
        trim_pretty_midi(self.pm)
        for i,instrument in enumerate(self.pm.instruments):
            for n,note in enumerate(instrument.notes):
                self.schedule.enter(note.start, 1, midiout.send_message,
                                    ([0x90 + instrument.program, note.pitch, note.velocity],))
                self.schedule.enter(note.end, 1, midiout.send_message,
                                    ([0x80 + instrument.program, note.pitch, 0],))
        self.schedule.run()
        return self.pm

    def stop(self):
        self.schedule.stop()
        if self.pm:
            for instrument in self.pm.instruments:
                for note in instrument.notes:
                    midiout.send_message(
                        [0x80 + instrument.program, note.pitch, 0])

threaded_player = ThreadedPlayer()


In [6]:
# # save all MIDI to CSV metadata

# import pandas as pd

# fns = list(sorted(list_files('output')))

# results = []
# for fn in fns:
#     pm = pretty_midi.PrettyMIDI(fn)
#     metadata = guess_metadata(pm.instruments[0].notes)
#     metadata['filename'] = fn
#     results.append(metadata)

# df = pd.DataFrame(results)
# df.to_csv('metadata.csv', index=False)

In [7]:
import json
from collections import defaultdict

review_data = defaultdict(dict)
with open('review_data.json', 'r') as f:
    review_data.update(json.load(f))

In [8]:
css_widget = widgets.HTML("""
<style>
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
.cell-output-ipywidget-background pre {
   color: white !important;
   font-family: Menlo;
}
.cell-output-ipywidget-background button {
   color: white !important;
   background-color: transparent !important;
   font-size: 64px;
   height: 1.6em;
   width: fit-content;
   font-family: Menlo;
}
</style>
""")
display(css_widget)

HTML(value='\n<style>\n.cell-output-ipywidget-background {\n   background-color: transparent !important;\n}\n.…

In [9]:

# df.sort_values(by='velocity_median')['filename'].values
# df.sort_values(by='velocity_median')

In [12]:
global i, fn

df = pd.read_csv('metadata.csv')

fns = list(df.sort_values(by='median_polyphony', ascending=False)['filename'].values)
# import random
# random.shuffle(fns)

# fns = [fn for fn in review_data if 'status' in review_data[fn] and review_data[fn]['status'] == 'star']
fns = [fn for fn in review_data if 'status' in review_data[fn] and review_data[fn]['status'] != 'drop']

i = -1
fn = None

skip_backward_button = Button(description="⏮️")
repeat_button = Button(description="🔁")
skip_forward_button = Button(description="⏭️")
stop_button = Button(description="🛑")
star_button = Button(description="✨⭐️✨")
save_button = Button(description="👍")
drop_button = Button(description="👎")
output_widget = widgets.Output()

def load_idx():
    global i, fn

    if i == -1:
        i = 0
    fn = fns[i]

    if 'listens' in review_data[fn]:
        review_data[fn]['listens'] += 1
    else:
        review_data[fn]['listens'] = 1
    review_data[fn]['last_listen'] = time.time()

    with output_widget:
        clear_output()
        pm = threaded_player.play(fn)
        print(i, fn)
        show_pr(pm)

def skip(n):
    global i
    valid = False
    while not valid:
        i += n
        if i < 0:
            i += len(fns)
        if i >= len(fns):
            i -= len(fns)
        fn = fns[i]
        # if fn not in review_data or \
        #     'status' not in review_data[fn]:
        #     valid = True
        #     break
        valid = True
    load_idx()

def set_label(label):
    global i, fn
    if fn is not None:
        review_data[fn]['status'] = label
    if label == 'drop':
        skip(+1)

skip_backward_button.on_click(lambda x: skip(-1))
repeat_button.on_click(lambda x: load_idx())
skip_forward_button.on_click(lambda x: skip(+1))
stop_button.on_click(lambda x: threaded_player.stop())
star_button.on_click(lambda x: set_label('star'))
save_button.on_click(lambda x: set_label('save'))
drop_button.on_click(lambda x: set_label('drop'))

display(HBox([skip_backward_button, repeat_button, skip_forward_button, stop_button, star_button, save_button, drop_button]))
display(output_widget)

Output()

In [ ]:
fns

In [187]:
with open('review_data.json', 'w') as f:
    json.dump(review_data, f)

In [190]:
# output/2023-05-12/dear-chocolate-material.mid # beethoven
# output/2023-05-12/insurance-standard-percentage.mid # beethoven
# output/2023-05-11/serve-stable-map.mid # swear this is something??
# output/2023-05-11/mine-background-administration.mid # can't remember who this is
# output/2023-05-12/excitement-stroke-prompt.mid # river flows in you
len(review_data), len(fns)

(277, 2848)

In [ ]:
midiout.close_port()